In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings 
warnings.filterwarnings('ignore')

## Records.csv

In [ ]:
df_records = pd.read_csv('records_record.csv')

In [ ]:
df_records = df_records.sort_values(by = ['user_id', 'date']).reset_index(drop = True)

In [ ]:
DW_COLS = list(df_records.columns)

DW_COLS.remove('user_id')
DW_COLS.insert(1, 'user_id')

df_record = df_records[DW_COLS]

In [ ]:
df_record

In [ ]:
df_record.alcohol_blackout.value_counts().index

In [ ]:
df_record.alcohol_blackout.value_counts()

## Amber's DataFrame

In [ ]:
df_record_HM = pd.read_csv('records_record (1).csv')

In [ ]:
df_record_HM

In [ ]:
df_record_HM.user_id.unique()

In [ ]:
df_record_HM = df_record_HM.sort_values(by = ['user_id', 'date']).reset_index(drop = True)
df_record_HM = df_record_HM[DW_COLS]

In [ ]:
df_record_HM_DW = df_record_HM[df_record_HM.user_id.isin(df_record.user_id.unique())].reset_index(drop = True)

In [ ]:
df_record_HM_DW

## DW vs Original

In [ ]:
df_record.alcohol_desire.value_counts().sort_index()

In [ ]:
df_record_HM_DW.alcohol_desire.value_counts().sort_index()

## Check User_ID Null

In [ ]:
# Match Index with the ID Column
df_record = df_record.set_index('id')
df_record_HM_DW = df_record_HM_DW.set_index('id')

mask = df_record['user_id'].isna()

df_record.loc[mask] = df_record_HM_DW.loc[df_record.loc[mask].index]
df_record = df_record.reset_index()

In [ ]:
df_record.user_id.isna().sum()

In [ ]:
df_record.alcohol_blackout.value_counts().index

In [ ]:
df_record.alcohol_desire = df_record.alcohol_desire.astype('int')

In [ ]:
df_record.alcohol_desire.value_counts()

In [ ]:
df_record_HM_DW.alcohol_desire.value_counts()

## Check Null Values in Features

In [ ]:
df_record.isna().sum()

In [ ]:
df_record.meal_morning.fillna(0, inplace = True)
df_record.meal_lunch.fillna(0, inplace = True)
df_record.meal_dinner.fillna(0, inplace = True)
df_record.meal_night.fillna(0, inplace = True)

In [ ]:
df_record.isna().sum()

In [ ]:
df_record.columns

In [ ]:
alcohol_cols = ['alcohol_amount_beer_glass', 'alcohol_amount_beer_can', 'alcohol_amount_beer_bottle',
       'alcohol_amount_soju_glass', 'alcohol_amount_soju_bottle',
       'alcohol_amount_wine_glass', 'alcohol_amount_wine_bottle',
       'alcohol_amount_liquor_glass', 'alcohol_amount_liquor_bottle',
       'alcohol_amount_rice_bowl', 'alcohol_amount_rice_bottle']

In [ ]:
df_record[alcohol_cols]= df_record[alcohol_cols].fillna(0)

In [ ]:
df_record.isna().sum()

In [ ]:
df_record.alcohol_with_whom.fillna('other', inplace = True)
df_record.alcohol_how_long.fillna('other', inplace = True)
df_record.alcohol_where.fillna('etc', inplace = True)
df_record.alcohol_blackout.fillna('other', inplace = True)

df_record.drop(columns = ['alcohol_why_etc'], inplace = True)

In [ ]:
df_record.isna().sum().sum()

In [ ]:
df_record.alcohol_desire.value_counts().sort_index()

In [ ]:
df_record.date = df_record.date.str.replace('-', '/')
df_record.date = pd.to_datetime(df_record.date)

In [ ]:
df_record

In [ ]:
fall_dt = pd.to_datetime(df_record.date.dt.strftime('%Y-%m-%d') + ' ' + df_record.sleep_fall_at.astype(str))
wake_dt = pd.to_datetime(df_record.date.dt.strftime('%Y-%m-%d') + ' ' + df_record.sleep_wake_at.astype(str))

In [ ]:
fall_dt_corrected = fall_dt.where(fall_dt <= wake_dt, fall_dt - pd.Timedelta(days = 1)) # Move One Day Earlier if the condition is false

In [ ]:
df_record['sleep_minute'] = (wake_dt - fall_dt_corrected) / pd.Timedelta(minutes=1)

## Preprocess Alcohol Amount

### Beer

In [ ]:
df_record.alcohol_amount_beer_bottle = df_record.alcohol_amount_beer_bottle * (500 / 200)
df_record.alcohol_amount_beer_can = df_record.alcohol_amount_beer_can * (355 / 200)

df_record['amount_beer_glass'] = df_record.alcohol_amount_beer_glass + df_record.alcohol_amount_beer_can + df_record.alcohol_amount_beer_bottle 

In [ ]:
df_record.amount_beer_glass = round(df_record.amount_beer_glass, 2)

### Soju

In [ ]:
df_record.alcohol_amount_soju_bottle = df_record.alcohol_amount_soju_bottle * (360 / 50)

df_record['amount_soju_glass'] = df_record.alcohol_amount_soju_glass + df_record.alcohol_amount_soju_bottle
df_record.amount_soju_glass = round(df_record.amount_soju_glass, 2)

### Wine

In [ ]:
df_record.alcohol_amount_wine_bottle = df_record.alcohol_amount_wine_bottle * (750 / 200)

df_record['amount_wine_glass'] = df_record.alcohol_amount_wine_glass + df_record.alcohol_amount_wine_bottle

df_record.amount_wine_glass = round(df_record.amount_wine_glass, 2)

### Whiskey

In [ ]:
df_record.alcohol_amount_liquor_bottle = df_record.alcohol_amount_liquor_bottle * (700 / 30)

df_record['amount_whiskey_glass'] = df_record.alcohol_amount_liquor_glass + df_record.alcohol_amount_liquor_bottle

df_record.amount_whiskey_glass = round(df_record.amount_whiskey_glass, 2)

### Rice Beer

In [ ]:
df_record.alcohol_amount_rice_bottle *= (750 / 200)
df_record['amount_rice_glass'] = df_record.alcohol_amount_rice_bottle  + df_record.alcohol_amount_rice_bowl
df_record.amount_rice_glass = round(df_record.amount_rice_glass, 2)

## Dummy Encoding

In [ ]:
df_record.columns

In [ ]:
dummy_cols = ['alcohol_amount_etc', 'alcohol_why_stress', 'alcohol_why_social',
       'alcohol_why_happy', 'alcohol_why_sleep', 'alcohol_why_lonely',
       'alcohol_why_unavoidable', 'alcohol_why_habit', 'alcohol_why_pairing',
       'alcohol_with_whom', 'alcohol_how_long',
       'alcohol_where', 'alcohol_blackout'
    
]

In [ ]:
df_dummy = df_record.copy()

In [ ]:
df_dummy = pd.get_dummies(
    df_dummy,
    columns = dummy_cols,
    drop_first = True
)

In [ ]:
df_dummy.columns

In [ ]:
df_dummy

In [ ]:
df_dummy.to_csv('PrepApp1.csv', index= False)